# One-click Colab hand-off

Run all cells on a fresh Colab runtime. This notebook exercises the repository
on a real Colab host and publishes a factual summary. It is **not** a benchmark
scorer:

- It reads OSCD **training** cities and training labels only.
- It never unzips, reads or scores the official test labels.
- It does not train, fine-tune, run v3, or modify `model.onnx` / `best_model.pt`.
- No result exists until this notebook is run. Do not cite its output before it
  appears in `benchmarks/results/COLAB_RUN_SUMMARY.md` on `main`.

Expected Drive layout:

```text
My Drive/oscd/
  archive (1).zip        # OSCD images and training labels
  weights/
    model.onnx           # verified Drive copy, expected 33,268,410 bytes
    best_model.pt        # optional here; copied if present but not used
```

The separate `test_labels.zip` is intentionally not used.

In [ ]:
from pathlib import Path
from datetime import date
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
from urllib.parse import quote, urlsplit, urlunsplit

DRIVE_DIR = Path('/content/drive/MyDrive/oscd')
REPO_URL = 'https://github.com/Aravind-Subramanian-16/satellite-change-detection.git'
REPO_DIR = Path('/content/repo')
OSCD_DIR = Path('/content/oscd')

ARCHIVE = DRIVE_DIR / 'archive (1).zip'
MODEL_SOURCE = DRIVE_DIR / 'weights' / 'model.onnx'
BEST_MODEL_SOURCE = DRIVE_DIR / 'weights' / 'best_model.pt'
EXPECTED_MODEL_BYTES = 33_268_410

IMAGES_ROOT = OSCD_DIR / 'images' / 'Onera Satellite Change Detection dataset - Images'
TRAIN_LABELS_ROOT = OSCD_DIR / 'train_labels' / 'Onera Satellite Change Detection dataset - Train Labels'
RESULTS_DIR = REPO_DIR / 'benchmarks' / 'results'
DRIVE_RESULTS = DRIVE_DIR / 'results'

print('[setup 0/4] Paths configured; no OSCD test labels are referenced.')

In [ ]:
print('[setup 1/4] Mounting Google Drive...')
from google.colab import drive
drive.mount('/content/drive')

if not ARCHIVE.is_file():
    raise FileNotFoundError(f'Missing OSCD archive: {ARCHIVE}')

print('[setup 2/4] Unzipping OSCD images and training labels...')
OSCD_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(['unzip', '-q', '-o', str(ARCHIVE), '-d', str(OSCD_DIR)], check=True)
if not IMAGES_ROOT.is_dir():
    raise FileNotFoundError(f'Missing images directory: {IMAGES_ROOT}')
if not TRAIN_LABELS_ROOT.is_dir():
    raise FileNotFoundError(f'Missing training-labels directory: {TRAIN_LABELS_ROOT}')

print('[setup 3/4] Cloning or updating the repository...')
if REPO_DIR.exists():
    if not (REPO_DIR / '.git').is_dir():
        raise RuntimeError(f'{REPO_DIR} exists but is not a git checkout')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

required_repo_files = (
    REPO_DIR / 'benchmarks' / 'oscd_loco_thresholds.py',
    REPO_DIR / 'benchmarks' / 'colab_run_summary.py',
)
missing_repo_files = [str(path.relative_to(REPO_DIR)) for path in required_repo_files if not path.is_file()]
if missing_repo_files:
    raise RuntimeError(
        'The remote checkout is missing Step 7 files: '
        + ', '.join(missing_repo_files)
        + '. Push the latest main before running this notebook.'
    )

if not MODEL_SOURCE.is_file():
    raise FileNotFoundError(f'Missing Drive model: {MODEL_SOURCE}')
shutil.copy2(MODEL_SOURCE, REPO_DIR / 'model.onnx')
if BEST_MODEL_SOURCE.is_file():
    shutil.copy2(BEST_MODEL_SOURCE, REPO_DIR / 'best_model.pt')
else:
    print('best_model.pt is not present on Drive; it is not used by this notebook.')

print('[setup 4/4] Installing benchmark dependencies without exact onnx/numpy pins...')
requirement_lines = []
for name in ('requirements.txt', 'requirements-benchmark.txt'):
    for line in (REPO_DIR / name).read_text().splitlines():
        stripped = line.strip()
        if stripped.startswith('-r '):
            continue
        if re.match(r'^(onnx|numpy)\s*==', stripped):
            continue
        requirement_lines.append(line)
Path('/content/req_colab.txt').write_text('\n'.join(requirement_lines) + '\n')
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-r', '/content/req_colab.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'pytest'], check=True)
print('onnxruntime and numpy ranges are kept; only exact onnx/numpy pins would be removed.')

sys.path.insert(0, str(REPO_DIR))
from benchmarks.oscd_eval import OFFICIAL_TEST_CITIES, read_city_list

train_list = IMAGES_ROOT / 'train.txt'
if not train_list.is_file():
    raise FileNotFoundError(f'Missing train list: {train_list}')
train_cities = read_city_list(train_list)
if len(train_cities) != 14:
    raise ValueError(f'Expected 14 training cities, found {len(train_cities)}')
leaked = sorted(set(train_cities) & set(OFFICIAL_TEST_CITIES))
if leaked:
    raise ValueError(f'Official test cities appear in train.txt: {leaked}')
if 'test' in str(TRAIN_LABELS_ROOT).lower():
    raise ValueError(f'Refusing to use a test-label path: {TRAIN_LABELS_ROOT}')
print(f'[setup] Verified {len(train_cities)} training cities; no official test city is in train.txt.')
print(f'[setup] Training labels: {TRAIN_LABELS_ROOT}')

In [ ]:
print('[a/4] Running the test suite...')
pytest_run = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q'],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
pytest_output = pytest_run.stdout + pytest_run.stderr
print(pytest_output, end='')
pytest_summary_line = next(
    (
        line.strip()
        for line in reversed(pytest_output.splitlines())
        if any(token in line for token in (' passed', ' failed', ' error', ' skipped'))
    ),
    '',
)
if not pytest_summary_line:
    raise RuntimeError('pytest emitted no summary line')
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
(DRIVE_RESULTS / 'pytest_summary.txt').write_text(pytest_summary_line + '\n')
print(f'[a/4] pytest exit code {pytest_run.returncode}: {pytest_summary_line}')
if pytest_run.returncode != 0:
    raise RuntimeError('pytest failed; fix the failure before running LOCO or publishing.')

In [ ]:
print('[b/4] Running train-only leave-one-city-out threshold robustness...')
loco_path = RESULTS_DIR / 'oscd_loco_thresholds.json'
loco_run = subprocess.run(
    [
        sys.executable,
        str(REPO_DIR / 'benchmarks' / 'oscd_loco_thresholds.py'),
        '--root', str(OSCD_DIR),
        '--output', str(loco_path),
    ],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
print(loco_run.stdout, end='')
print(loco_run.stderr, end='')
if loco_run.returncode != 0:
    raise RuntimeError('LOCO script failed; no summary was published.')

loco_report = json.loads(loco_path.read_text())
if loco_report.get('result_status') != 'computed_on_training_cities_only':
    raise ValueError('LOCO report is not marked training-only')
if loco_report['guards']['official_test_cities_in_train_list']:
    raise ValueError('LOCO report contains an official test city in train.txt')
if loco_report['guards']['test_labels_path_accepted'] is not False:
    raise ValueError('LOCO report does not record the test-label guard')
for family, values in loco_report['families'].items():
    pooled = values['pooled_held_out']
    print(
        f"[b/4] {family}: pooled held-out F1={pooled['f1']:.4f} "
        f"IoU={pooled['iou']:.4f} distinct={values['distinct_selected_configs']}"
    )
print('[b/4] Training-only guard confirmed; no test city was scored.')

In [ ]:
print('[c/4] Checking the Drive model and onnxruntime loading...')
try:
    import onnxruntime as ort
    ort_import_error = None
except Exception as exc:
    ort = None
    ort_import_error = f'{type(exc).__name__}: {exc}'

model_path = REPO_DIR / 'model.onnx'
digest = hashlib.sha256()
with MODEL_SOURCE.open('rb') as handle:
    for chunk in iter(lambda: handle.read(1 << 20), b''):
        digest.update(chunk)

model_check = {
    'path': 'model.onnx',
    'source': '$DRIVE_DIR/weights/model.onnx',
    'size_bytes': MODEL_SOURCE.stat().st_size,
    'expected_bytes': EXPECTED_MODEL_BYTES,
    'sha256': digest.hexdigest(),
    'loads': False,
    'input_names': [],
    'providers': [],
    'error': None,
}
if ort is None:
    model_check['error'] = ort_import_error
else:
    try:
        session = ort.InferenceSession(str(model_path), providers=['CPUExecutionProvider'])
        model_check['loads'] = True
        model_check['input_names'] = [item.name for item in session.get_inputs()]
        model_check['providers'] = list(session.get_providers())
    except Exception as exc:
        model_check['error'] = f'{type(exc).__name__}: {exc}'

(RESULTS_DIR / 'colab_model_check.json').write_text(json.dumps(model_check, indent=2) + '\n')
print(f"[c/4] size={model_check['size_bytes']:,} expected={EXPECTED_MODEL_BYTES:,}")
print(f"[c/4] sha256={model_check['sha256']}")
print(f"[c/4] onnxruntime_loads={model_check['loads']} inputs={model_check['input_names']}")
if not model_check['loads']:
    print(f"[c/4] WARNING: model did not load: {model_check['error']}")
if model_check['size_bytes'] != EXPECTED_MODEL_BYTES:
    print('[c/4] WARNING: model size differs from the expected Drive copy.')

The next cell reads the three saved artifacts written above. It does not
recompute metrics, invent values, or read test labels.

In [ ]:
print('[d/4] Writing benchmarks/results/COLAB_RUN_SUMMARY.md...')
from benchmarks.colab_run_summary import write_colab_summary

commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
summary_path = RESULTS_DIR / 'COLAB_RUN_SUMMARY.md'
summary_text = write_colab_summary(
    loco_path=RESULTS_DIR / 'oscd_loco_thresholds.json',
    model_check_path=RESULTS_DIR / 'colab_model_check.json',
    pytest_summary_path=DRIVE_RESULTS / 'pytest_summary.txt',
    commit=commit,
    pytest_exit_code=pytest_run.returncode,
    output_path=summary_path,
    generated_date=date.today().isoformat(),
)
print('```text')
print(summary_text)
print('```')
print(f'[d/4] wrote {summary_path}')

In [ ]:
print('[push] Publishing only the summary and small result files...')
summary_files = [
    'benchmarks/results/COLAB_RUN_SUMMARY.md',
    'benchmarks/results/oscd_loco_thresholds.json',
    'benchmarks/results/colab_model_check.json',
]
github_token = None
token_remote = None
push_env = None
try:
    try:
        from google.colab import userdata
        github_token = userdata.get('GITHUB_TOKEN')
    except Exception:
        github_token = None

    if not github_token:
        print('Push skipped: copy the block above')
    else:
        subprocess.run(['git', 'add', *summary_files], cwd=REPO_DIR, check=True)
        staged = subprocess.check_output(
            ['git', 'status', '--porcelain', '--', *summary_files],
            cwd=REPO_DIR,
            text=True,
        )
        if not staged.strip():
            print('No new summary or result files to push.')
        else:
            subprocess.run(
                [
                    'git',
                    '-c', 'user.name=Colab Runner',
                    '-c', 'user.email=colab-runner@users.noreply.github.com',
                    'commit', '-m', 'results: Colab training-split robustness and model check',
                ],
                cwd=REPO_DIR,
                check=True,
                capture_output=True,
                text=True,
            )
            remote = subprocess.check_output(
                ['git', 'remote', 'get-url', 'origin'], cwd=REPO_DIR, text=True
            ).strip()
            parts = urlsplit(remote)
            if parts.scheme not in ('https', 'http') or not parts.hostname:
                raise RuntimeError('origin is not an HTTP(S) remote')
            netloc = f'x-access-token:{quote(github_token, safe="")}@{parts.hostname}'
            if parts.port:
                netloc += f':{parts.port}'
            token_remote = urlunsplit((parts.scheme, netloc, parts.path, parts.query, parts.fragment))
            push_env = os.environ.copy()
            push_env['GIT_CONFIG_COUNT'] = '1'
            push_env['GIT_CONFIG_KEY_0'] = 'remote.origin.url'
            push_env['GIT_CONFIG_VALUE_0'] = token_remote
            subprocess.run(
                ['git', 'push', 'origin', 'main'],
                cwd=REPO_DIR,
                env=push_env,
                check=True,
                capture_output=True,
                text=True,
            )
            print('Pushed the summary and small result files to main.')
except Exception:
    print('Push skipped: copy the block above')
finally:
    github_token = None
    token_remote = None
    push_env = None

## After the run

- If the summary printed above, copy that block and tell the project owner
  **"Colab done."** The owner will pull `main`, fill any pending cells from the
  saved summary, and rebuild the paper figures and claims audit.
- If the last cell printed `Push skipped: copy the block above`, nothing was
  uploaded. Copy the summary block manually; do not paste a token anywhere.
- This notebook does not score official test cities, train a model, run v3, or
  establish an accuracy claim. Do not describe its output as a test result.
- `COLAB_RUN_SUMMARY.md` is generated from saved JSON and the real pytest line.
  It is not pre-written and does not exist until this run.